In [1]:
import os
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# 1. Securely load the API key from the .env file
load_dotenv(override=True)
API_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
SYMBOL = "HDB"

def fetch_historical_baseline():
    print(f"Fetching historical market data for {SYMBOL}...")
    
    url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={SYMBOL}&apikey={API_KEY}"
    response = requests.get(url)
    data = response.json()
    
    time_series = data.get("Time Series (Daily)", {})
    
    # Convert JSON to Pandas DataFrame
    df = pd.DataFrame.from_dict(time_series, orient="index")
    df = df.rename(columns={"4. close": "close_price"})
    df = df[["close_price"]].astype(float) # We only need the closing price
    
    # Clean up the dates and sort chronologically (oldest to newest)
    df.index.name = "trade_date"
    df = df.reset_index()
    df["trade_date"] = pd.to_datetime(df["trade_date"])
    df = df.sort_values("trade_date").reset_index(drop=True)
    
    # 2. THE MATH ENGINE: Calculate Bollinger Bands
    # We use a 20-day rolling window, which is the institutional standard for risk monitoring
    df["20_day_SMA"] = df["close_price"].rolling(window=20).mean()
    df["20_day_STD"] = df["close_price"].rolling(window=20).std()
    
    # Calculate the Lower Risk Band (SMA - 2x Standard Deviation)
    # If the price drops below this line, it represents a statistically significant crash (anomaly)
    df["lower_risk_band"] = df["20_day_SMA"] - (2 * df["20_day_STD"])
    
    # Round everything for clean reporting
    df = df.round(2)
    
    # Drop the first 19 days (since a 20-day rolling average needs 20 days to calculate)
    df = df.dropna()
    
    # Save the baseline model
    df.to_csv("hdfc_risk_baseline.csv", index=False)
    
    print("✅ Success! Mathematical baseline established.")
    print("\nMost Recent 5 Days of Risk Data:")
    print(df.tail())

if __name__ == "__main__":
    fetch_historical_baseline()

Fetching historical market data for HDB...
✅ Success! Mathematical baseline established.

Most Recent 5 Days of Risk Data:
   trade_date  close_price  20_day_SMA  20_day_STD  lower_risk_band
95 2026-09-22        23.41       22.91        0.50            21.91
96 2026-09-23        22.79       22.86        0.47            21.93
97 2026-09-24        22.83       22.85        0.46            21.93
98 2026-09-25        23.01       22.88        0.45            21.97
99 2026-09-28        22.41       22.84        0.46            21.92


In [3]:
import pandas as pd

# ==========================================
# PHASE 2: LIVE ANOMALY DETECTION ENGINE
# ==========================================

# 1. Load the baseline data we saved in Phase 1
df = pd.read_csv("hdfc_risk_baseline.csv")
df["trade_date"] = pd.to_datetime(df["trade_date"]) # Revert string back to datetime

# 2. Extract the most recent baseline metrics (acting as our "yesterday" threshold)
latest_data = df.iloc[-1]
baseline_date = latest_data["trade_date"].strftime('%Y-%m-%d')
lower_risk_band = latest_data["lower_risk_band"]

print("--- 🛡️ ACTIVE RISK MONITORING ENGINE ---")
print(f"Baseline Date: {baseline_date}")
print(f"Critical Lower Risk Band (Floor): ${lower_risk_band}")

# 3. SIMULATE LIVE MARKET DATA
# Let's pretend it is mid-day trading today and HDB suddenly crashes to $20.50
today_live_price = 20.50 

print(f"\n📡 Incoming Live Price: ${today_live_price}")

# 4. THE TRIGGER LOGIC
if today_live_price < lower_risk_band:
    # Calculate the severity of the anomaly
    deviation = round(lower_risk_band - today_live_price, 2)
    
    print("\n🚨 FATAL ANOMALY DETECTED 🚨")
    print(f"Action: Asset has breached the lower Bollinger Band by ${deviation}!")
    print("Status: Initiating automated executive alert sequence...")
    
    # TODO: Add Email or Slack delivery code here
    
else:
    print("\n✅ Asset is trading within normal statistical bounds. No action required.")

--- 🛡️ ACTIVE RISK MONITORING ENGINE ---
Baseline Date: 2026-09-28
Critical Lower Risk Band (Floor): $21.92

📡 Incoming Live Price: $20.5

🚨 FATAL ANOMALY DETECTED 🚨
Action: Asset has breached the lower Bollinger Band by $1.42!
Status: Initiating automated executive alert sequence...


In [7]:
import json
import os
import requests
from dotenv import load_dotenv

load_dotenv(override=True)
SLACK_WEBHOOK_URL = os.getenv("SLACK_WEBHOOK_URL")

# --- EXECUTE THE TRIGGER ---
if today_live_price < lower_risk_band:
  deviation = round(lower_risk_band - today_live_price, 2)

  slack_message = {
      "text": (
          f"🚨 *ALGORITHMIC RISK ALERT: {SYMBOL}* 🚨\n\n*Live Price:*"
          f" ${today_live_price}\n*Risk Floor (Lower Band):*"
          f" ${lower_risk_band}\n*Deviation:* Crashed ${deviation} below"
          " statistical norms.\n\n_Immediate quantitative review required._"
      )
  }

  response = requests.post(
      SLACK_WEBHOOK_URL,
      data=json.dumps(slack_message),
      headers={"Content-Type": "application/json"},
  )

  if response.status_code == 200:
    print("✅ Alert successfully delivered to Slack #risk-alerts channel.")
  else:
    print(f"❌ Failed to send Slack alert. Error code: {response.status_code}")

✅ Alert successfully delivered to Slack #risk-alerts channel.
